# PadDx_miR_3. BPC

# 앞부분: Utility + Pool 생성

In [1]:
pip install pandas biopython


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from padlock_design.utils import *
from padlock_design.filters import *
from padlock_design.assembler import *
from padlock_design.thermo import *
from padlock_design.reporter import *

In [3]:
import random
import pandas as pd
import pickle

random.seed(15)  

def generat_pool(spacer_priorities, cas12a_priorities):
    spacer_pools = {L: generate_candidates([L], num_samples=500000, gc_range=(35,60), detailed_filter=spacer_filter) for L in spacer_priorities}
    cas12a_pools = {L: generate_candidates([L], num_samples=500000, gc_range=(35,65), detailed_filter=cas12a_filter) for L in cas12a_priorities}

    print("Spacer pools:", {k: len(v) for k, v in spacer_pools.items()}, "→ Total:", sum(len(v) for v in spacer_pools.values()))
    print("Cas12a pools:", {k: len(v) for k, v in cas12a_pools.items()}, "→ Total:", sum(len(v) for v in cas12a_pools.values()))
    
    try:
        with open("pools_by_length.pkl", "wb") as f:
            pickle.dump((spacer_pools, cas12a_pools), f)
    except Exception as e:
        print(f"[ERROR] Failed to save pools: {e}")

    return spacer_pools, cas12a_pools


if __name__ == "__main__":
    # Define target lengths based on Padlock probe design requirements
    spacer_length = [16, 15, 17]
    cas12a_length = [20, 21, 22]

    # Execute high-throughput library generation
    spacer_candiates, cas12a_candiates = generat_pool(spacer_length, cas12a_length)


Spacer pools: {16: 7407, 15: 16735, 17: 2305} → Total: 26447
Cas12a pools: {20: 14994, 21: 7317, 22: 4191} → Total: 26502


# 뒷부분: Validation + Padlock Assembly 전체 후보 스크리닝

In [4]:
def assemble_and_filter_padlocks(name, seq, spacer_pools, cas12a_pools, num_to_generate=30):
    """
    Main pipeline to find the best split and generate validated padlock candidates.
    """
    target_name = name
    target_seq = seq
    
    split_res = find_best_split(target_name, target_seq)
    print_split_report(split_res)   
    
    # Setup Validation Parameters
    VALIDATION_PARAMS = {
        # 01: Poly-N repeats (e.g., AAAAA)
        "has_long_homopolymer(padlock_seq)": 4,

        # 02: Global self-complementarity (Hairpin risk)
        "has_self_complementarity_global(padlock_seq)": 6,

        # 03: Tandem repeats (e.g., ATATAT)
        "has_tandem_repeat(padlock_seq)": 5,

        # 04: Internal structure within modules
        "has_self_complementarity(spacer)": 5,
        "has_self_complementarity(cas12a_core)": 5,

        # 05: Cross-hybridization between modules
        "has_cross_hybridization(spacer, cas12a_core)": 5,
        "has_cross_hybridization(spacer_rc, cas12a_core)": 5,
        
        # 06: Cross-hybridization with miRNA arms
        "has_cross_hybridization(mirna_left_rc, spacer)": 3,
        "has_cross_hybridization(mirna_right_rc, cas12a_core)": 3,

        # 07: Global vs. Module cross-reactivity
        "has_cross_hybridization(padlock_seq, spacer)": 5,
        "has_cross_hybridization(padlock_seq, cas12a_core)": 5,
        "has_cross_hybridization(padlock_seq, mirna_left_rc)": 6,
        "has_cross_hybridization(padlock_seq, mirna_right_rc)": 6,

        # 08: RCA product vs. Original target/spacer
        "has_cross_hybridization(padlock_seq_rc, mirna_seq)": 6,
    }

    results = validate_padlocks(
        spacer_pools, 
        cas12a_pools, 
        split_res, 
        VALIDATION_PARAMS, 
        NUM_PADLOCKS=num_to_generate, 
        seed_offset=0
    )
    
    return results 

if __name__ == "__main__":
    # Input miRNA information
    target_miRNA = "hsa-miR-21-5p"
    target_sequence = "UAGCUUAUCAGACUGAUGUUGA"
    
    # Run pipeline
    filtered_candidates = assemble_and_filter_padlocks(
        target_miRNA, 
        target_sequence, 
        spacer_candiates, 
        cas12a_candiates, 
        num_to_generate = 10 # Target number of safe sequences
    )
    
    if filtered_candidates:
        filtered_df = pd.DataFrame(filtered_candidates)
        print(f"\n[SUCCESS] Selection of {len(filtered_df)} candidates is ready.")
    else:
        print("\n[WARNING] NO SAFE CANDIDATES WERE GENERATED. CHECK PARAMETERS.")


[Target: hsa-miR-21-5p]
  Optimal Split: 11 + 11 nt
  
  5' Arm: TAGCTTATCAG 
     (GC=36.4%, Tm=32.6°C) -> Padlock 3' end: CTGATAAGCTA
     
  3' Arm: ACTGATGTTGA 
     (GC=36.4%, Tm=36.9°C) -> Padlock 5' end: TCAACATCAGT
  ------------------------------------------------------------
[Padlock 1 / 10] Generated. (Attempts: 212)
[Padlock 2 / 10] Generated. (Attempts: 671)
[Padlock 3 / 10] Generated. (Attempts: 1228)
[Padlock 4 / 10] Generated. (Attempts: 3660)
[Padlock 5 / 10] Generated. (Attempts: 4013)
[Padlock 6 / 10] Generated. (Attempts: 4835)
[Padlock 7 / 10] Generated. (Attempts: 5004)
[Padlock 8 / 10] Generated. (Attempts: 6310)
[Padlock 9 / 10] Generated. (Attempts: 9570)
[Padlock 10 / 10] Generated. (Attempts: 9738)

[SUCCESS] Selection of 10 candidates is ready.


1. 생성되는 padlock보면 spacer, cas = 16, 20 가 전부임
2. miRNA arm이랑 padlock 조건을 6이하로 내리면 너무 안나옴 

# 생성된 후보 Padlock 및 구성요소 검토 코드

In [ ]:
import pandas as pd
from padlock_design.reporter import print_detailed_analysis

def run_padlock_validation_suite(dataframe):
    """
    Validates physical and thermodynamic properties for all padlock candidates.

    Args:
        dataframe (pd.DataFrame): DataFrame containing candidate sequences and metadata.
    """
    # Add Candidate ID to the first column if missing
    if 'Candidate ID' not in dataframe.columns:
        dataframe.insert(0, 'Candidate ID', range(1, len(dataframe) + 1))

    for i, row in dataframe.iterrows():
        padlock = row["Padlock Sequence (5'-P)"].replace("[5'-P]-", "")
        spacer_seq = row["Spacer (Pad)"]
        cas12a_core = row["Cas12a Core (Pad)"]

        print(f"\n{'='*90}")
        print(f"Candidate #{row['Candidate ID']} | Target: {row.get('Target Name', 'Unknown')}")
        print(f"{'='*90}")

        print_detailed_analysis("Full Padlock", padlock)
        print_detailed_analysis("Spacer", spacer_seq)
        print_detailed_analysis("Cas12a Core", cas12a_core)

if __name__ == "__main__":
    # Check if 'filtered_candidates' exists in the current scope
    if 'filtered_candidates' in locals():
        test_df = pd.DataFrame(filtered_candidates)
        run_padlock_validation_suite(test_df)
    else:
        print("[ERROR] No data found (filtered_candidates is missing).")


Candidate #1 | Target: hsa-miR-21-5p
  [Full Padlock]
    Seq    : CTGATAAGCTATGAGTGGAATACTACGAATCCAGGGTCAATAACCAGATTTCAACATCAGT
    Length : 61 nt
    GC%    : 39.34%
    Tm     : 75.97 °C
    ΔG     : -75.74 kcal/mol
--------------------------------------------------
  [Spacer]
    Seq    : TGAGTGGAATACTACG
    Length : 16 nt
    GC%    : 43.75%
    Tm     : 52.05 °C
    ΔG     : -19.43 kcal/mol
--------------------------------------------------
  [Cas12a Core]
    Seq    : AATCCAGGGTCAATAACCAG
    Length : 20 nt
    GC%    : 45.00%
    Tm     : 60.43 °C
    ΔG     : -25.09 kcal/mol
--------------------------------------------------

Candidate #2 | Target: hsa-miR-21-5p
  [Full Padlock]
    Seq    : CTGATAAGCTATGAGTGGAATACTACGCATTACGGACAAGAAGCAGTAAATCAACATCAGT
    Length : 61 nt
    GC%    : 39.34%
    Tm     : 76.11 °C
    ΔG     : -76.69 kcal/mol
--------------------------------------------------
  [Spacer]
    Seq    : TGAGTGGAATACTACG
    Length : 16 nt
    GC%    : 43.75%
    T

delta G 값 다르게 계산됨 -수정 필요

# delta G를 기준으로 정렬

In [6]:
def analyze_and_sort_padlocks(df, dg_threshold=-3.5):
    """
    Analyzes sequences and filters out candidates with ΔG <= threshold (too stable).
    Returns a filtered dataframe sorted by stability.
    """
    
    if 'Candidate ID' not in df.columns:
        df.insert(0, 'Candidate ID', range(1, len(df) + 1))
    
    results_struct = []
    results_dg = []

    print(f"\n{'[ DNA Padlock Batch Analysis & Filtering ]':^75}")
    print(f"{'ID':<4} | {'Target Name':<15} | {'ΔG (kcal/mol)':<15} | {'Status'}")
    print("-" * 85)

    for idx, row in df.iterrows():
        raw_seq = str(row["Padlock Sequence (5\'-P)"])
        clean_seq = raw_seq.replace("[5'-P]-", "").strip()
        
        struct, dg = run_rnafold_dna(clean_seq)
        
        results_struct.append(struct)
        results_dg.append(dg)
        
        status = "PASS" if dg > dg_threshold else "REJECTED (Too Stable)"
        print(f"{row['Candidate ID']:<4} | {row['Target Name']:<15} | {dg:>10.2f}     | {status}")

    df['Calculated ΔG'] = results_dg
    df['Secondary Structure'] = results_struct
    
    initial_count = len(df)
    filtered_df = df[df['Calculated ΔG'] > dg_threshold].copy()
    rejected_count = initial_count - len(filtered_df)
    
    # Sort by stability (highest ΔG first = least folded)
    sorted_df = filtered_df.sort_values(by='Calculated ΔG', ascending=False).reset_index(drop=True)
    
    print("-" * 85)
    print(f"Filtering complete!")
    print(f" - Total Processed: {initial_count}")
    print(f" - Rejected (ΔG < {dg_threshold}): {rejected_count}")
    print(f" - Survivors: {len(sorted_df)}")
    print("-" * 85 + "\n")
    
    return sorted_df

if __name__ == "__main__":
    sorted_results = analyze_and_sort_padlocks(test_df, dg_threshold= -3)
    
    with pd.option_context(
        "display.max_rows", None,        
        "display.max_columns", None,     
        "display.max_colwidth", None     
    ):
        display(sorted_results)


                [ DNA Padlock Batch Analysis & Filtering ]                 
ID   | Target Name     | ΔG (kcal/mol)   | Status
-------------------------------------------------------------------------------------
1    | hsa-miR-21-5p   |      -2.80     | PASS
2    | hsa-miR-21-5p   |      -2.30     | PASS
3    | hsa-miR-21-5p   |      -4.10     | REJECTED (Too Stable)
4    | hsa-miR-21-5p   |      -2.00     | PASS
5    | hsa-miR-21-5p   |      -1.10     | PASS
6    | hsa-miR-21-5p   |      -0.50     | PASS
7    | hsa-miR-21-5p   |      -3.30     | REJECTED (Too Stable)
8    | hsa-miR-21-5p   |      -1.60     | PASS
9    | hsa-miR-21-5p   |      -1.30     | PASS
10   | hsa-miR-21-5p   |      -4.20     | REJECTED (Too Stable)
-------------------------------------------------------------------------------------
Filtering complete!
 - Total Processed: 10
 - Rejected (ΔG < -3): 3
 - Survivors: 7
-------------------------------------------------------------------------------------



,Candidate ID,Target Name,Padlock Sequence (5'-P),Padlock Length,Padlock Tm,Padlock ΔG (kcal/mol),GC% (Padlock),Spacer (Pad),Spacer Tm,Cas12a Core (Pad),Cas12a Tm,Short spacer,miRNA Sequence,miRNA 5' Arm (Pad),miRNA 5' Arm len,miRNA 3' Arm (Pad),miRNA 3' Arm len,Calculated ΔG,Secondary Structure
0,6,hsa-miR-21-5p,[5'-P]-CTGATAAGCTACACTAAGATTTGCGTAAATAACCGAACCCAGTCCCTTATATCAACATCAGT,62,75.92,-77.07,38.71,CACTAAGATTTGCGTA,50.82,AATAACCGAACCCAGTCCCT,64.26,TATA,TAGCTTATCAGACTGATGTTGA,CTGATAAGCTA,11,TCAACATCAGT,11,-0.5,..................((((....))))................................
1,5,hsa-miR-21-5p,[5'-P]-CTGATAAGCTACATAACGGTGACAAACCCTACCATCGGTCTCAAGAGTAATCAACATCAGT,61,77.46,-78.17,42.62,CATAACGGTGACAAAC,52.83,CCTACCATCGGTCTCAAGAG,62.45,TAA,TAGCTTATCAGACTGATGTTGA,CTGATAAGCTA,11,TCAACATCAGT,11,-1.1,.................(((.....)))..(((...)))......................
2,9,hsa-miR-21-5p,[5'-P]-CTGATAAGCTAAGTTGTAAGGTAAACCAACGACAATCCGAACCTCAGAATTTCAACATCAGT,62,76.37,-77.98,38.71,AGTTGTAAGGTAAACC,50.08,AACGACAATCCGAACCTCAG,63.34,AATT,TAGCTTATCAGACTGATGTTGA,CTGATAAGCTA,11,TCAACATCAGT,11,-1.3,(((((.......((((.(((((..................))))......).))))))))).
3,8,hsa-miR-21-5p,[5'-P]-CTGATAAGCTAGATTGTAAAGGTCCTGGCTACCCTCAAGAGTCCCTTATATCAACATCAGT,61,77.14,-76.79,42.62,GATTGTAAAGGTCCTG,51.26,GCTACCCTCAAGAGTCCCTT,64.34,ATA,TAGCTTATCAGACTGATGTTGA,CTGATAAGCTA,11,TCAACATCAGT,11,-1.6,.(((((((((((............))))))......(((.....))).)))))........
4,4,hsa-miR-21-5p,[5'-P]-CTGATAAGCTAGTTCTCGTAACCAAATCACCGACCTCTTAGCGAAGTTATATCAACATCAGT,62,76.39,-78.11,40.32,GTTCTCGTAACCAAAT,51.06,CACCGACCTCTTAGCGAAGT,64.99,TATA,TAGCTTATCAGACTGATGTTGA,CTGATAAGCTA,11,TCAACATCAGT,11,-2.0,(((((..((((....(((............))).....))))...(((.....)))))))).
5,2,hsa-miR-21-5p,[5'-P]-CTGATAAGCTATGAGTGGAATACTACGCATTACGGACAAGAAGCAGTAAATCAACATCAGT,61,76.11,-76.69,39.34,TGAGTGGAATACTACG,52.05,CATTACGGACAAGAAGCAGT,61.15,AAA,TAGCTTATCAGACTGATGTTGA,CTGATAAGCTA,11,TCAACATCAGT,11,-2.3,(((((..((((((.((((....)))).)))...........))).((......))))))).
6,1,hsa-miR-21-5p,[5'-P]-CTGATAAGCTATGAGTGGAATACTACGAATCCAGGGTCAATAACCAGATTTCAACATCAGT,61,75.97,-75.74,39.34,TGAGTGGAATACTACG,52.05,AATCCAGGGTCAATAACCAG,60.43,ATT,TAGCTTATCAGACTGATGTTGA,CTGATAAGCTA,11,TCAACATCAGT,11,-2.8,(((((((.((.....((((..........)))).(((.....))))).)).....))))).


나중에 추가하고 싶은거
1. 비슷한 delta G값 끼리 어디에 2차구조 생기는지? ex) spacer > cas > miRNA arm 순으로  
2. 2차 구조 (( )) 개수 적은거 ?
3. -3.5kcal/mol 넘으면 제거하기?4
4. target name + candidate ID 합치기

# NCBI 전체 후보 검증 코드 (API)

## NCBI BLAST Nucleotide Database Guide

### 1. High-Quality Curated Databases (RefSeq Series)
These databases are curated and managed directly by NCBI staff, offering the highest level of reliability and sequence accuracy.

* **RefSeq Select RNA sequences (`refseq_select`)**
    * Contains the **single most representative transcript** for each gene.
    * *Best use:* Initial searches when you want to avoid redundant results from multiple isoforms.
* **Reference RNA sequences (`refseq_rna`)**
    * Includes all curated RNA sequences, encompassing a wider range of **transcript variants (isoforms)** than the 'Select' version.
* **RefSeq Reference genomes (`refseq_reference_genomes`)**
    * Consists of the **standardized "Gold Standard" genome sequences** that represent a specific species.
* **RefSeq Genome Database (`refseq_genomes`)**
    * A broader collection including both the reference genomes and all other high-quality curated genome data available in RefSeq.
* **Human RefSeqGene sequences (`RefSeq_Gene`)**
    * Specifically focuses on **human genomic gene sequences**.
    * *Best use:* Clinical genetics, medical research, and precise variant analysis within human gene loci.

---

### 2. Comprehensive & Broad Databases
These databases are designed for maximum coverage across all known biological sequences.

* **Nucleotide collection (`nr/nt`)**
    * **The "Standard" Default:** This is the most comprehensive database, aggregating sequences from GenBank, RefSeq, PDB, and others.
    * *Best use:* General homology searches or when looking for matches across diverse or less-studied organisms.


---

### Summary Table for Quick Selection

| Database | Level of Curation | Primary Use Case |
| :--- | :--- | :--- |
| **`refseq_select`** | High (Curated) | Simple, representative gene searches |
| **`refseq_rna`** | High (Curated) | Comprehensive transcript/isoform analysis |
| **`refseq_genomes`** | High (Curated) | Genome-wide mapping and comparison |
| **`nr/nt`** | Variable | Broad discovery and unknown sequence identification |

---

### ⚙️ Current Configuration & Customization
* **Default Database:** `refseq_rna` (Reference RNA sequences)
* **Default Organism:** *Homo sapiens* (Taxid: 9606)

> **Note:** The current pipeline is optimized for **human transcript analysis** using the RefSeq RNA database. If your research targets a different species (e.g., *Mus musculus*) or requires a broader search (e.g., `nr/nt`), you can easily adjust the `database` and `organism` parameters in the configuration section of the script.

In [8]:
from Bio.Blast import NCBIWWW, NCBIXML
import time

def run_blastn_and_parse(query_seq, label, db="refseq_rna", expect=100, word_size=7, hitlist_size=20, min_ident=0.90):
    """NCBI BLAST 실행 후 결과 파싱"""
    print(f"\n--- [{label}] | DB: {db}, txid9606[ORGN] ---")
    print("Query sequence:", query_seq)

    result_handle = NCBIWWW.qblast("blastn", db, query_seq,
                                   entrez_query="txid9606[ORGN]",
                                   expect=expect,
                                   word_size=word_size,
                                   hitlist_size=hitlist_size)
    blast_record = NCBIXML.read(result_handle)
    result_handle.close()

    hits = {"perfect": {"RNA": [], "Genomic": [], "Other": []},
            "partial": {"RNA": [], "Genomic": [], "Other": []},
            "contig":  {"RNA": [], "Genomic": [], "Other": []},
            "3prime":  {"RNA": [], "Genomic": [], "Other": []}}

    for alignment in blast_record.alignments:
        for hsp in alignment.hsps:
            subj = hsp.sbjct
            cat = classify_hit(alignment.title)

            # 완전 일치
            if hsp.identities == len(query_seq) and hsp.align_length == len(query_seq):
                hits["perfect"][cat].append(alignment.title)

            # 부분 일치 (90~99%)
            identity_ratio = hsp.identities / hsp.align_length
            if min_ident <= identity_ratio < 1.0 and hsp.align_length >= int(min_ident * len(query_seq)):
                hits["partial"][cat].append(
                    f"{alignment.title} (identity={identity_ratio:.2f}, mismatches={hsp.align_length-hsp.identities})"
                )

            # 연속 일치
            if check_contiguous_match(hsp.query, subj, min_match=15):
                hits["contig"][cat].append(alignment.title)

            # 3' 말단 일치
            if check_3prime_match(hsp.query, subj, min_3p=12):
                hits["3prime"][cat].append(alignment.title)

    return hits

def evaluate_candidate_safety(hits):
    """
    Analyzes BLAST hit results to determine rejection status.
    Note: 3' terminal 12nt match condition has been removed.
    """
    reasons = []
    
    # 1. Perfect Match Check
    for cat in hits["perfect"]:
        if hits["perfect"][cat]:
            reasons.append(f"PERFECT MATCH FOUND IN {cat}")

    # 2. Partial Match Check (High Identity > 90%)
    for cat in hits["partial"]:
        if hits["partial"][cat]:
            reasons.append(f"HIGH IDENTITY PARTIAL MATCH (>90%) IN {cat}")

    # 3. Contiguous Match Check (Consecutive 15nt)
    for cat in hits["contig"]:
        if hits["contig"][cat]:
            reasons.append(f"CONTIGUOUS MATCH (>= 15nt) IN {cat}")
    
    # 4. 3' Terminal Match (3' 말단 12nt 일치)
    for cat in hits["3prime"]:
        if hits["3prime"][cat]:
            reasons.append(f"Critical: 3' terminal match >= 12nt in {cat}")

    is_safe = len(reasons) == 0
    return is_safe, reasons

# =========================================================
# Execution Loop (Optimized: Cas12a-First Logic)
# =========================================================

passed_candidates = []
TARGET_COUNT = 5

for idx, row in sorted_results.iterrows():
    candidate_id = row.get('Candidate ID', idx + 1)
    print(f"\n{'='*30} [STARTING VALIDATION: CANDIDATE #{candidate_id}] {'='*30}")

    # STEP 1: Validate CAS12A Core first
    cas_name = f"CAS12A_CORE_#{candidate_id}"
    cas_seq = row['Cas12a Core (Pad)']
    
    print(f"▶ STEP 1: Running BLAST for {cas_name}...")
    cas_hits = run_blastn_and_parse(cas_seq, cas_name, db="refseq_rna")
    is_cas_safe, cas_reasons = evaluate_candidate_safety(cas_hits)

    if not is_cas_safe:
        print(f"\n[REJECTED] CANDIDATE #{candidate_id} FAILED AT CAS12A VALIDATION.")
        print(f"   - REASON(S): {', '.join(cas_reasons)}")
        # Skip Spacer validation and move to the next candidate immediately
        continue 

    # STEP 2: Validate SPACER Core only if CAS12A passed
    spacer_name = f"SPACER_#{candidate_id}"
    spacer_seq = row['Spacer (Pad)']
    
    print(f"▶ STEP 2: Running BLAST for {spacer_name}...")
    spacer_hits = run_blastn_and_parse(spacer_seq, spacer_name, db="refseq_rna")
    is_spacer_safe, spacer_reasons = evaluate_candidate_safety(spacer_hits)

    if not is_spacer_safe:
        print(f"\n[REJECTED] CANDIDATE #{candidate_id} FAILED AT SPACER VALIDATION.")
        print(f"   - REASON(S): {', '.join(spacer_reasons)}")
        continue

    # SUCCESS: Candidate passed all hurdles
    print(f"\n[SUCCESS] CANDIDATE #{candidate_id} IS VERIFIED AS SAFE.")
    passed_candidates.append(row)
    
    # --- EARLY EXIT LOGIC ---
    if len(passed_candidates) >= TARGET_COUNT:
        print(f"\n[INFO] TARGET COUNT ({TARGET_COUNT}) REACHED. TERMINATING VALIDATION PROCESS.")
        break
    # ------------------------

    # Rate limiting for NCBI API
    time.sleep(2)

print(f"\n{'='*20} [FINAL SUMMARY: {len(passed_candidates)} CANDIDATE(S) SURVIVED] {'='*20}")


============================== [STARTING VALIDATION: CANDIDATE #6] ==============================
▶ STEP 1: Running BLAST for CAS12A_CORE_#6...

--- [CAS12A_CORE_#6] | DB: refseq_rna, txid9606[ORGN] ---
Query sequence: AATAACCGAACCCAGTCCCT
▶ STEP 2: Running BLAST for SPACER_#6...

--- [SPACER_#6] | DB: refseq_rna, txid9606[ORGN] ---
Query sequence: CACTAAGATTTGCGTA

[SUCCESS] CANDIDATE #6 IS VERIFIED AS SAFE.

============================== [STARTING VALIDATION: CANDIDATE #5] ==============================
▶ STEP 1: Running BLAST for CAS12A_CORE_#5...

--- [CAS12A_CORE_#5] | DB: refseq_rna, txid9606[ORGN] ---
Query sequence: CCTACCATCGGTCTCAAGAG

[REJECTED] CANDIDATE #5 FAILED AT CAS12A VALIDATION.
   - REASON(S): CONTIGUOUS MATCH (>= 15nt) IN RNA, Critical: 3' terminal match >= 12nt in RNA

============================== [STARTING VALIDATION: CANDIDATE #9] ==============================
▶ STEP 1: Running BLAST for CAS12A_CORE_#9...

--- [CAS12A_CORE_#9] | DB: refseq_rna, txid9606[ORG

In [ ]:
pip install openpyxl

In [ ]:
from padlock_design.reporter import export_candidate_results

if __name__ == "__main__":
    final_df = export_candidate_results(passed_candidates)
    
    if final_df is not None:
        display(final_df)


[SUCCESS] 2 CANDIDATES EXPORTED TO EXCEL.
[SUCCESS] PATH: results\validated_padlocks_20260127_1816.xlsx


,Candidate ID,Target Name,Padlock Sequence (5'-P),Padlock Length,Padlock Tm,Padlock ΔG (kcal/mol),GC% (Padlock),Spacer (Pad),Spacer Tm,Cas12a Core (Pad),Cas12a Tm,Short spacer,miRNA Sequence,miRNA 5' Arm (Pad),miRNA 5' Arm len,miRNA 3' Arm (Pad),miRNA 3' Arm len,Calculated ΔG,Secondary Structure
0,6,hsa-miR-21-5p,[5'-P]-CTGATAAGCTACACTAAGATTTGCGTAAATAACCGAACC...,62,75.92,-77.07,38.71,CACTAAGATTTGCGTA,50.82,AATAACCGAACCCAGTCCCT,64.26,TATA,TAGCTTATCAGACTGATGTTGA,CTGATAAGCTA,11,TCAACATCAGT,11,-0.5,..................((((....))))...................
4,4,hsa-miR-21-5p,[5'-P]-CTGATAAGCTAGTTCTCGTAACCAAATCACCGACCTCTT...,62,76.39,-78.11,40.32,GTTCTCGTAACCAAAT,51.06,CACCGACCTCTTAGCGAAGT,64.99,TATA,TAGCTTATCAGACTGATGTTGA,CTGATAAGCTA,11,TCAACATCAGT,11,-2.0,(((((..((((....(((............))).....))))...(...
